# Scenario model: cheapest port and route under strait closures
For every origin region (Asia via Shanghai, Europe via Rotterdam) and every Saudi destination, the model compares all ports in Saudi Arabia, the UAE and Oman:

**sea leg** (origin → port, with chokepoint closures) + **port** (handling, dwell) + **road leg** (port → destination, with border delays)

and picks the option with the lowest generalised cost per TEU (money plus the value of cargo time).

Inputs come from the repository: `config/costs.csv` (cost parameters, mostly placeholders marked `dummy`), `config/ports.csv`, `config/border_crossings.csv`, `config/destinations.csv`, and the weekly `data-latest` release (OSM roads, PortWatch). Change a value in a config file and rerun; no code changes needed.

In [ ]:
!pip -q install osmnx searoute osmium
!git clone -q --depth 1 https://github.com/danaalsh/jeddah-port-logistics.git repo 2>/dev/null || git -C repo pull -q

## 0. Settings

In [ ]:
import sys
from pathlib import Path

REPO = Path("repo")
sys.path.insert(0, str(REPO))
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/jeddah_port")
except ImportError:  # running outside Colab
    OUT_DIR = Path("out")
DATA_DIR, CACHE_DIR = OUT_DIR / "data", OUT_DIR / "cache"
for d in (OUT_DIR, DATA_DIR, CACHE_DIR):
    d.mkdir(parents=True, exist_ok=True)
GPKG = OUT_DIR / "jeddah_port_gis.gpkg"   # same GeoPackage as notebook 01; adds sim_* layers
SITES_CSV = OUT_DIR / "sites.csv"         # optional extra destinations (same columns as config/destinations.csv)
REFRESH_DATA = True                        # download the latest release files (weekly)

## 1. Data and road network

In [ ]:
import pandas as pd
from sim import data, network, places, model

files = data.fetch(["gcc_major.osm.pbf", "gcc_extras.osm.pbf", "osm_date.txt",
                    "portwatch_ports_list.csv", "portwatch_ports_daily_SAU.csv"], DATA_DIR, refresh=REFRESH_DATA)
OSM_DATE = files["osm_date.txt"].read_text().strip()

p = model.load_params(REPO / "config/costs.csv")
G = network.build_graph(files["gcc_major.osm.pbf"], CACHE_DIR, truck_max_kph=int(p["truck_max_kph"]))  # ~5 min first time
crossings = pd.read_csv(REPO / "config/border_crossings.csv")
print(network.add_border_delays(G, crossings).to_string(index=False))
vot = network.set_costs(G, p)
print(f"{len(G.nodes):,} junctions, {len(G.edges):,} road segments; OSM snapshot {OSM_DATE}")
print(f"value of cargo time: {vot:.2f} USD per TEU-hour")

## 2. Ports and destinations

In [ ]:
ports = model.load_ports(REPO / "config/ports.csv", files["portwatch_ports_list.csv"])
dests = [pd.read_csv(REPO / "config/destinations.csv"), places.industrial_cities(files["gcc_extras.osm.pbf"])]
if SITES_CSV.exists():
    dests.append(pd.read_csv(SITES_CSV))
dests = pd.concat(dests, ignore_index=True)
assert dests.dest_id.is_unique, "dest_id values must be unique"
print(f"{len(ports)} ports, {len(dests)} destinations")
dests[["dest_id", "name", "type", "lat", "lon"]]

## 3. Road legs (port → destination)
Cheapest truck route from every port to every destination. Independent of the sea scenarios, so computed once.

In [ ]:
roads = model.road_legs(G, ports, dests)
print(f"{sum(r is not None for r in roads.values())}/{len(roads)} port-destination pairs reachable by road")
print("port distance to nearest major road (m):",
      {k: round(r["port_snap_m"]) for (k, _), r in roads.items() if r is not None})

## 4. Scenarios
`model.SCENARIOS` maps each scenario to the sea passages it closes. Add your own, e.g. `model.SCENARIOS["suez_closed"] = ["suez"]`.

In [ ]:
runs = [model.run(sc, ports, dests, roads, p, vot) for sc in model.SCENARIOS]
options = pd.concat([o for o, _ in runs], ignore_index=True)
sea_legs = pd.concat([s for _, s in runs], ignore_index=True)
best = model.compare(options[options["rank"] == 1].reset_index(drop=True))
used = set(zip(best.scenario, best.origin, best.port_key))  # sea lanes some destination actually uses
sea_legs["used"] = [k in used for k in zip(sea_legs.scenario, sea_legs.origin, sea_legs.port_key)]

# Which port serves each destination, by scenario (Asia origin)
best[best.origin == "Asia (Shanghai)"].pivot(index="dest_name", columns="scenario", values="port_key")[list(model.SCENARIOS)]

In [ ]:
# Cost and time change against the baseline (Asia origin), largest increases first
cols = ["scenario", "dest_name", "port_key_base", "port_key", "total_usd_base", "total_usd", "extra_usd",
        "total_days_base", "total_days", "extra_days", "borders"]
best[(best.origin == "Asia (Shanghai)") & (best.scenario != "baseline")][cols].sort_values("extra_usd", ascending=False)

## 5. Check against what happened (Hormuz, from 1 March 2026)
PortWatch recorded Saudi Gulf ports losing most of their imports after Hormuz closed. Does the model move the same traffic? The model can only be compared on Saudi ports; trucks entering via the UAE do not show in PortWatch's Saudi port data.

In [ ]:
pw = pd.read_csv(files["portwatch_ports_daily_SAU.csv"], parse_dates=["date"])
SHOCK = pd.Timestamp("2026-03-01")
end = (pw.date.max() + pd.Timedelta(days=1)).to_period("M").to_timestamp()  # complete months only

def monthly(a, b):
    # mean monthly imports per port over months that have data (a failed download leaves a month empty)
    w = pw[(pw.date >= a) & (pw.date < b)]
    return w.groupby(["portid", w.date.dt.to_period("M")])["import"].sum().groupby("portid").mean()


obs = (100 * (monthly(SHOCK, end) / monthly(SHOCK - pd.DateOffset(months=12), SHOCK) - 1)).round(0)
print(f"observed: monthly imports {SHOCK:%b %Y} to {end - pd.Timedelta(days=1):%b %Y} vs the 12 months before")

asia = best[best.origin == "Asia (Shanghai)"]
served = lambda sc: asia[asia.scenario == sc].port_key.value_counts()
check = pd.DataFrame({"port": ports.port_key, "portwatch_id": ports.portwatch_id})
check["observed_import_change_pct"] = check.portwatch_id.map(obs)
check["model_destinations_baseline"] = check.port.map(served("baseline")).fillna(0).astype(int)
check["model_destinations_hormuz_closed"] = check.port.map(served("hormuz_closed")).fillna(0).astype(int)
check[check.port.isin(["JED", "KAP", "DMM", "JUB", "KFK", "SOH"])]

## 6. Save to the GeoPackage

In [ ]:
import geopandas as gpd

pts = lambda df: gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df.lon, df.lat), crs="EPSG:4326")
model.save(GPKG, {
    "sim_ports": pts(ports),
    "sim_destinations": pts(dests),
    "sim_sea_legs": gpd.GeoDataFrame(sea_legs, geometry="geometry", crs="EPSG:4326"),
    "sim_road_legs": model.road_geometries(best, roads),
    "sim_best": best.assign(osm_date=OSM_DATE),
    "sim_options": options,
})
best.to_csv(OUT_DIR / "sim_best.csv", index=False)
print("Saved sim_* layers to", GPKG)

## 7. Map
Baseline and Hormuz-closed routes for the Asia origin: sea legs dashed, road legs solid.

In [ ]:
import matplotlib.pyplot as plt

road_lines = model.road_geometries(best, roads)
seas = gpd.GeoDataFrame(sea_legs, geometry="geometry", crs="EPSG:4326")
fig, axes = plt.subplots(1, 2, figsize=(14, 6.5), sharex=True, sharey=True)
for ax, sc, color in zip(axes, ["baseline", "hormuz_closed"], ["#2a78d6", "#eb6834"]):
    used = best[(best.scenario == sc) & (best.origin == "Asia (Shanghai)")].port_key.unique()
    s = seas[(seas.scenario == sc) & (seas.origin == "Asia (Shanghai)") & seas.port_key.isin(used)]
    s.plot(ax=ax, color=color, linewidth=1, linestyle="--")
    road_lines[(road_lines.scenario == sc) & (road_lines.origin == "Asia (Shanghai)")].plot(ax=ax, color=color, linewidth=1.4)
    pts(ports).plot(ax=ax, color="black", markersize=14, zorder=5)
    pts(dests).plot(ax=ax, color="#52514e", markersize=8, zorder=5)
    for _, r in ports.iterrows():
        ax.annotate(r.port_key, (r.lon, r.lat), fontsize=7, xytext=(3, 3), textcoords="offset points")
    ax.set_xlim(34, 62); ax.set_ylim(14, 32); ax.set_title(sc.replace("_", " ").capitalize(), loc="left")
    ax.set_axis_off()
fig.suptitle("Cheapest routes from Asia to Saudi destinations", x=0.01, ha="left", fontweight="bold")
plt.tight_layout(); plt.show()